<p align="center">
<a href="https://duckietown.com"><img src="../assets/images/dtlogo.png" alt="Duckietown Logo" width="30%"></a>
</p>


# Inspect a Local ROS 2 Graph

The previous notebook built the supplied package and confirmed that ROS 2 could find its publisher and subscriber executables. The next step is to start the publisher and investigate how its running behavior appears in the graph.

Which node does the executable create? Is anything subscribed to its topic, and do messages arrive at the expected rate?

This notebook connects the publisher’s terminal output to its discovered interfaces, then observes how subscribing changes the graph and measures the arriving messages.

## Start the installed publisher

The commands run in the dedicated environment described in the [README](../README.md#prerequisites), using the package built in the previous notebook. Earlier exercise publishers should be stopped so their endpoints do not contribute to the observations.

In Terminal 1, load the distribution and workspace overlay:

```bash
source /opt/ros/${ROS2_DISTRO}/setup.bash
source "$COLCON_WS/install/setup.bash"
```

Successful sourcing normally produces no output. The installed publisher can then be started by its registered executable name:

```bash
ros2 run ros2_pubsub my_publisher
```

The publisher should begin logging messages such as:

```text
[INFO] [1792158813.325050256] [talker]: Hello from ROS 2!
[INFO] [1792158814.319314122] [talker]: Hello from ROS 2!
```

The timestamps will differ, while the initial message text should remain `Hello from ROS 2!`. The lines continue roughly once per second, showing that the publisher’s callback is executing.

The executable name was `my_publisher`, whereas the log identifies the node as `talker`. The next observation checks that name in the graph. Leave Terminal 1 running while Terminal 2 performs the inspection.

## Find the node and its topic

In Terminal 2, load the same environment:

```bash
source /opt/ros/${ROS2_DISTRO}/setup.bash
source "$COLCON_WS/install/setup.bash"
```

These commands normally return silently. Listing the nodes should then reveal the publisher:

```bash
ros2 node list
```

```text
/talker
```

If the listing remains empty while publication continues, the direct-discovery check introduced in notebook 4 helps distinguish an observation problem from a stopped publisher.

The topic listing identifies the interfaces visible alongside that node:

```bash
ros2 topic list -t
```

With only the exercise publisher running, the result is:

```text
/chatter [std_msgs/msg/String]
/parameter_events [rcl_interfaces/msg/ParameterEvent]
/rosout [rcl_interfaces/msg/Log]
```

The application publishes text on `/chatter`, while the other entries support parameter events and logging.

To find out how many endpoints currently use `/chatter`, inspect the topic:

```bash
ros2 topic info /chatter
```

The initial observation shows:

```text
Type: std_msgs/msg/String
Publisher count: 1
Subscription count: 0
```

One publisher is present, and no subscriptions have joined yet. Publishing can continue in this state because the publisher does not require a subscriber before producing messages.

## Observe messages and the subscription they create

The previous notebooks used `ros2 topic echo` to receive messages. Keeping that observer running now makes its subscription visible to another inspection command.

In Terminal 2:

```bash
ros2 topic echo /chatter std_msgs/msg/String
```

The explicit type matches the topic listing and allows the observer to create its subscription without first inferring the message type from discovery.

Messages should begin arriving:

```text
data: Hello from ROS 2!
---
data: Hello from ROS 2!
---
```

Each `---` ends one displayed message. The command continues receiving, so Terminal 2 remains occupied until `Ctrl-C` stops it.

To inspect the graph while both publisher and observer remain active, open Terminal 3 and load the environment:

```bash
source /opt/ros/${ROS2_DISTRO}/setup.bash
source "$COLCON_WS/install/setup.bash"
ros2 topic info /chatter
```

After the normally silent sourcing commands, the topic information should now include one subscription:

```text
Type: std_msgs/msg/String
Publisher count: 1
Subscription count: 1
```

The observer has become part of the graph it inspects. Its displayed messages establish that this subscription is receiving the publisher’s data.

Stop the echo command with `Ctrl-C` in Terminal 2, then repeat the topic inspection in Terminal 3:

```bash
ros2 topic info /chatter
```

After discovery reflects the change, the count returns to:

```text
Type: std_msgs/msg/String
Publisher count: 1
Subscription count: 0
```

Terminal 1 continues publishing throughout this change.

For a finite observation, Terminal 2 can request just one message:

```bash
ros2 topic echo --once /chatter std_msgs/msg/String
```

```text
data: Hello from ROS 2!
---
```

After receiving the message, the command exits and its temporary subscription disappears.

<!--
Proposed checkpoint replacement
ID: single-message-observation
Source: #observe-messages-and-the-subscription-they-create
Question: Continuous echo changes the subscription count from zero to one. How can a single message be received without leaving that observer running?
Model answer: Run ros2 topic echo --once /chatter std_msgs/msg/String. It creates a temporary subscription, receives and displays one message, then exits. Discovery may take a short time to reflect the subscription's removal.
-->

## Connect the endpoint to its delivery settings

The brief topic inspection reports counts of publishers and subscribers. Its verbose form adds information about each endpoint, including the node that owns it and its quality-of-service settings:

```bash
ros2 topic info --verbose /chatter
```

With the echo command stopped, selected lines from the publisher endpoint are:

```text
Node name: talker
Node namespace: /
Topic type: std_msgs/msg/String
```

```text
Endpoint type: PUBLISHER
```

```text
QoS profile:
  Reliability: RELIABLE
  History (Depth): UNKNOWN
  Durability: VOLATILE
```

The node name connects this endpoint to `/talker`. The QoS fields describe its delivery policies: `RELIABLE` supports retrying delivery, while `VOLATILE` supplies no retained-message history for subscribers that join later.

The observer therefore receives newly published messages after joining. The [ROS 2 QoS documentation](https://docs.ros.org/en/jazzy/Concepts/Intermediate/About-Quality-of-Service-Settings.html) explains these policies and their compatibility rules.

If `History (Depth)` appears as `UNKNOWN`, as in this example, the inspection provides no usable history-depth value.

The familiar interface command can separately inspect the message’s fields:

```bash
ros2 interface show std_msgs/msg/String
```

Omitting the introductory comments, the definition contains:

```text
string data
```

The definition explains why the received message has a text field named `data`, while the verbose topic observation identifies the endpoint supplying it.

<!--
Proposed checkpoint replacement
ID: message-type-inspection
Source: #connect-the-endpoint-to-its-delivery-settings
Question: Which observation identifies the publisher and its delivery policies, and which command reveals the fields of its String messages?
Model answer: ros2 topic info --verbose /chatter identifies the discovered publisher endpoint and its QoS policies. ros2 interface show std_msgs/msg/String displays the installed message definition, including string data.
-->

## Measure the arriving message rate

Receiving one message establishes delivery, but an application may also depend on how regularly new messages arrive.

In Terminal 2:

```bash
ros2 topic hz /chatter
```

This command creates a subscription and estimates the rate from received messages. After discovery and enough received samples, it should begin reporting the observed arrival rate: 

```text
average rate: 0.999
	min: 1.000s max: 1.003s std dev: 0.00139s window: 3
average rate: 1.000
	min: 0.997s max: 1.003s std dev: 0.00205s window: 5
```

An initial discovery warning may appear before measurements begin. Subsequent measurements indicate that messages are arriving. If the warning persists without measurements, check that the publisher remains active and that the topic is visible.

An average near `1.000` corresponds to approximately one received message per second. The minimum and maximum values describe observed intervals between messages, while the standard deviation summarizes their variation.

These measurements describe reception by this observer, so scheduling and system load can affect them. Small timing differences are expected even when the publisher requests a fixed rate.

Stop the measurement with `Ctrl-C` before continuing.

<!--
Proposed additional checkpoint
ID: observed-message-rate
Source: #measure-the-arriving-message-rate
Question: The publisher is configured for one message per second, and ros2 topic hz reports an average rate of 0.999. How should that result be interpreted?
Model answer: The observer receives messages at approximately the configured rate. Small differences can arise from scheduling and measurement timing. The command measures arrival rate at its subscription.
-->

### A note on latency

Alongside arrival rate, a robotics application needs to consider how old its data is when it becomes available for processing. Messages can arrive regularly while carrying information acquired earlier. A timestamp allows an observer to measure that age and relate it to latency along the data path. 

This timestamp-based measurement requires attention to clock settings and synchronization across computers. The `String` messages on `/chatter` contain only text, so a small stamped-message example will provide the additional field.

In Terminal 3, inspect the definition:

```bash
ros2 interface show geometry_msgs/msg/PointStamped
```

Omitting the introductory comment, the definition contains:

```text
std_msgs/Header header
	builtin_interfaces/Time stamp
		int32 sec
		uint32 nanosec
	string frame_id
Point point
	float64 x
	float64 y
	float64 z
```

The message combines a three-dimensional point with a timestamp and reference-frame name. A fixed demonstration point is sufficient here because the changing timestamp supplies the measurement.

In Terminal 3:

```bash
ros2 topic pub --rate 1 --node-name stamped_publisher \
  /latency_demo geometry_msgs/msg/PointStamped \
  "{header: {stamp: now, frame_id: map}, point: {x: 1.0, y: 0.0, z: 0.0}}"
```

The special value `now` tells the command-line publisher to refresh the timestamp for each publication. `map` labels the example’s coordinate frame; the point is synthetic.

The terminal should print `publisher: beginning loop`, followed by numbered publications whose timestamps change approximately once per second. Leave this publisher running while Terminal 2 observes it.

```bash
ros2 topic echo --once /latency_demo geometry_msgs/msg/PointStamped
```

The received message should have this structure, with a different timestamp:

```text
header:
  stamp:
    sec: 1792162459
    nanosec: 513182151
  frame_id: map
point:
  x: 1.0
  y: 0.0
  z: 0.0
---
```

After the command returns, measure the elapsed time from that timestamp to the observer’s receiving callback:

```bash
ros2 topic delay /latency_demo
```

Allow several messages to arrive. In this local environment, the output may resemble:

```text
average delay: 0.001
	min: 0.000s max: 0.001s std dev: 0.00006s window: 8
```

The reported durations are in seconds. A displayed average of `0.001` corresponds to approximately one millisecond. Because the values are rounded, `0.000` can represent a small positive delay rather than exactly zero. Results vary with system load and scheduling.

The tool calculates:

```text
delay = time at the receiving callback − header timestamp
```

Because this example refreshes the timestamp shortly before publication, the result includes the elapsed time through publication, transport, queuing, and scheduling of the CLI subscriber’s callback. Measuring latency experienced by an application would require the same calculation in that application’s receiving callback.

For sensor data, a timestamp may instead identify acquisition time, in which case the measured age also includes processing before publication. The timestamp’s meaning determines which part of the data path is being measured.

Both processes in this example use the same computer’s clock. Across computers, clock synchronization becomes necessary because an offset between clocks also appears in the subtraction. Both processes must use the same time basis, including when simulation time is enabled.

Stop the delay measurement with `Ctrl-C` in Terminal 2, then stop the stamped publisher with `Ctrl-C` in Terminal 3. Leave the original `/talker` publisher in Terminal 1 running for the next inspection.

<!--
Proposed additional checkpoint
ID: message-latency
Source: #measure-how-old-an-arriving-message-is
Question: A topic arrives at a steady 10 Hz, while its messages are two seconds old when received. What do the rate and delay measurements reveal?
Model answer: The observer receives messages regularly, approximately every 0.1 seconds, but their timestamps indicate a two-second age at reception. Interpreting that age requires knowing where the timestamps were assigned and whether the publisher and observer clocks agree.
-->

## Inspect the node’s other interfaces

Topic inspection starts from a stream. To examine the participating node as a whole:

```bash
ros2 node info /talker
```

The publisher section includes:

```text
Publishers:
  /chatter: std_msgs/msg/String
  /parameter_events: rcl_interfaces/msg/ParameterEvent
  /rosout: rcl_interfaces/msg/Log
```

The full result also lists service servers, including parameter-related services, and empty action sections.

The service names and types can be inspected directly:

```bash
ros2 service list -t
```

Among the returned entries are:

```text
/talker/get_parameters [rcl_interfaces/srv/GetParameters]
/talker/list_parameters [rcl_interfaces/srv/ListParameters]
/talker/set_parameters [rcl_interfaces/srv/SetParameters]
```

These are excerpts from the full listing. They expose operations associated with the node’s configuration. The available parameter names can then be listed:

```bash
ros2 param list /talker
```

In this environment, the listing should include:

```text
  start_type_description_service
  use_sim_time
```

The publisher’s Python constants, such as its publication rate, become remotely configurable parameters only if the implementation exposes them through that interface.

Finally:

```bash
ros2 action list -t
```

With only this exercise running, the command should return an empty listing because the publisher exposes no action interface.

## Finish the observation

With the echo and rate-measurement commands stopped, stop the publisher in Terminal 1 using `Ctrl-C`.

A direct discovery check allows time for the graph to reflect its departure:

```bash
ros2 node list --no-daemon --spin-time 5
```

With only the exercise nodes previously running, the listing should now be empty. Other exercises may leave additional nodes visible.

## Further reading

The ROS 2 Jazzy [CLI tutorials](https://docs.ros.org/en/jazzy/Tutorials/Beginner-CLI-Tools.html) cover graph inspection, while the [QoS documentation](https://docs.ros.org/en/jazzy/Concepts/Intermediate/About-Quality-of-Service-Settings.html) explains delivery policies.

## Checkpoint

The next cell contains the interactive self-check. A response must be entered or selected before the corresponding answer can be revealed.

## Summary and next steps

Starting the installed executable created `/talker` and its `/chatter` publisher. Echoing the topic added a visible subscription and revealed its messages, while rate measurement established how regularly those messages arrived. The stamped-message example also measured message age at the observer, showing why timestamps and a shared time basis are needed to interpret delay. Endpoint and node inspection connected those observations to delivery settings and other available interfaces.

[Notebook 8: Inspect a Duckiedrone ROS 2 Graph](./8-inspect-a-duckiedrone-ros-2-graph.ipynb) provides an optional application of these tools to telemetry. The dedicated-environment path continues with [Notebook 9: ROS 2 Node Lifecycles and Publishers](./9-ros-2-node-lifecycles-and-publishers.ipynb), which examines the implementation behind the publisher’s behavior.


In [ ]:
import sys
from pathlib import Path

working_directory = Path.cwd()
parent_directory = working_directory.parent
if (parent_directory / "packages").is_dir():
    parent_directory_path = str(parent_directory)
    sys.path.insert(0, parent_directory_path)

from packages.checkpoint_self_check import display_checkpoint_self_checks

display_checkpoint_self_checks()
